In [0]:
spark.sql('CREATE VOLUME IF NOT EXISTS data')

In [0]:
data_source = "s3://bucket-s3-for-databricks-free-edition/retail/"
dataset_retail = '/Volumes/workspace/default/data/retail'
data_catalog = 'workspace'

In [0]:
def path_exists(path):
  try:
    dbutils.fs.ls(path)
    return True
  except Exception as e:
    if 'java.io.FileNotFoundException' in str(e):
      return False
    else:
      raise

In [0]:
path_exists(dataset_retail)

In [0]:
def download_dataset(source, target):
    files = dbutils.fs.ls(source)
    for f in files:
        source_path = f"{source}/{f.name}"
        target_path = f"{target}/{f.name}"
        if not path_exists(target_path):
            print(f"Copying {f.name} ...")
            dbutils.fs.cp(source_path, target_path, True)

In [0]:
def get_index(dir):
    files = dbutils.fs.ls(dir)
    index = 0
    if files:
        file = max(files).name
        index = int(file.rsplit('.', maxsplit=1)[0])
    return index+1

In [0]:
def set_current_catalog(catalog_name):
    spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
# Structured Streaming
streaming_dir = f"{dataset_retail}/orders/parquet"
raw_dir = f"{dataset_retail}/orders/parquet-raw"

def load_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.parquet"
    print(f"Loading {latest_file} file to the retail dataset")
    dbutils.fs.cp(f"{streaming_dir}/{latest_file}", f"{raw_dir}/{latest_file}")

    
def load_new_data(all=False):
    index = get_index(raw_dir)
    if index >= 10:
        print("No more data to load\n")

    elif all == True:
        while index <= 10:
            load_file(index)
            index += 1
    else:
        load_file(index)
        index += 1

In [0]:
if not path_exists(dataset_retail):
    download_dataset(data_source, dataset_retail)

set_current_catalog(data_catalog)